In [ ]:

# If there's a GPU available...
import torch
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

Please make sure the outputs of above step confirms notebook to be in GPU mode. Else change it by going to Edit -->Notebook settings -->GPU (under Hardware acclerator drop down)

In [ ]:
INPUT_DIR ='/gdrive/MyDrive/BERT_Model'
# OUTPUT_DIR ='/gdrive/MyDrive/heirarchical_classifier_level_1/beauty'
# OUTPUT_DIR ='/gdrive/MyDrive/heirarchical_classifier_level_2'
OUTPUT_DIR ='/gdrive/MyDrive/heirarchical_classifier_level_0'

train_csv_path ='/gdrive/MyDrive/heirarchical_classifier_ML/data.csv'
# epochs = 4 #for level 1
epochs =5 #for level2 and 0

MAX_LEN =512

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
import os

In [ ]:
import pandas as pd

In [ ]:
!pip3 install transformers

In [ ]:
import torch
from transformers import BertModel, BertConfig, BertTokenizer, BertForSequenceClassification

In [ ]:
from keras.preprocessing.sequence import pad_sequences

In [ ]:
# model = BertModel.from_pretrained('bert-base-uncased')
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)


In [ ]:
# model.save_pretrained('/gdrive/MyDrive/BERT_Model')
# tokenizer.save_pretrained('/gdrive/MyDrive/BERT_Model')

In [ ]:
tokenizer_seq = BertTokenizer.from_pretrained(INPUT_DIR)

Preprocess class



In [ ]:
from nltk.tokenize import sent_tokenize
import hashlib

In [ ]:
import nltk
nltk.download('all')

In [ ]:
from math import ceil

In [ ]:
#preprocessing

class preprocessing:

  def __init__(self, df):
    self.df =df

  def preprocess(self, level, parent_class, present_class):
    weight_dict={}
    cat_label={}
    if(level ==0):
      unique_labels = self.df['Cat1'].unique()

      total_no =len(self.df.axes[0])

      count=0
      for el in unique_labels:
        # print(el)
        label_count = len(self.df[self.df['Cat1']==el].axes[0])
        # print(label_count)

        weight_dict[count] = total_no/label_count

        cat_label[el] =count
        count+=1


        # print('+'*70)

      self.df['Title'] = self.df['Title'].fillna('')
      self.df['Text'] = self.df['Text'].fillna('')

      X = list(self.df['Title']+'. '+self.df['Text'])
      Y = [cat_label[el] for el in list(self.df['Cat1'])]

    elif(level ==1):

      # if(present_class =='pet supplies' or present_class=='health personal care' or present_class=='beauty'):
      #   self.df_updated = self.df[self.df['Cat1'] ==present_class]

      #   unique_labels = self.df_updated['Cat2'].unique()

      #   total_no = len(self.df_updated.axes[0])

      #   count=0
      #   for el in unique_labels:
      #     print(el)

      #     label_count = len(self.df_updated[self.df_updated['Cat2']==el].axes[0])
      #     print(label_count)

      #     weight_dict[count] = total_no/label_count

      #     cat_label[el] =count
      #     count+=1

      #     print('.'*70)

      #   self.df_updated['Title'] = self.df_updated['Title'].fillna('')
      #   self.df_updated['Text'] = self.df_updated['Text'].fillna('')

      #   X = list(self.df_updated['Title']+'. '+self.df_updated['Text'])
      #   Y = [cat_label[el] for el in list(self.df_updated['Cat2'])]

      # elif(present_class =='grocery gourmet food' or present_class =='toys games' or present_class=='baby products'):
      self.df_updated = self.df[self.df['Cat1'] ==present_class]# filtering the primary category
      self.df_updated['Title'] = self.df_updated['Title'].fillna('')
      self.df_updated['Text'] = self.df_updated['Text'].fillna('')

      ls_new=[]
      for r in range(len(self.df_updated.axes[0])):
        di_temp = dict(self.df_updated.iloc[r])

        title =di_temp['Title']
        text = di_temp['Text']
        cat2 =di_temp['Cat2']

        st_hash = di_temp['productId']+di_temp['userId']+ str(di_temp['Time'])

        hash_ =hashlib.sha1(st_hash.encode('utf-8')).hexdigest()

        sent_list = sent_tokenize(text)

        for elm in sent_list:
          di_gran={}
          di_gran['hash'] =hash_
          di_gran['combined_text_granular'] =title+'. '+elm
          di_gran['Cat2'] = cat2

          ls_new.append(di_gran)


      self.df_new = pd.DataFrame(ls_new)

      unique_labels = self.df_new['Cat2'].unique()

      total_no = len(self.df_new.axes[0])

      count=0
      for el in unique_labels:
        print(el)

        label_count = len(self.df_new[self.df_new['Cat2']==el].axes[0])
        print(label_count)

        weight_dict[count] = total_no/label_count

        cat_label[el] =count
        count+=1

        print('.'*70)

      X = list(self.df_new['combined_text_granular'])
      Y = [cat_label[el] for el in list(self.df_new['Cat2'])]

    elif(level ==2):
      self.df['Title'] = self.df['Title'].fillna('')
      self.df['Text'] = self.df['Text'].fillna('')

      unique_labels = self.df['Cat3'].unique()

      ls_new =[]

      for tert_labels in unique_labels:

        df_updated = self.df[self.df['Cat3'] ==tert_labels]

        if(len(df_updated.axes[0]) <20):# to upscale category rows which are less than 20(by repeating as per their frequency)
          frequency = ceil(20/len(df_updated.axes[0]))

        else:
          frequency =1

        temp_cnt=0

        while(temp_cnt !=frequency):

          for r in range(len(df_updated.axes[0])):
            di_temp = dict(df_updated.iloc[r])


            title =di_temp['Title']
            text = di_temp['Text']
            cat3 =di_temp['Cat3']

            st_hash = di_temp['productId']+di_temp['userId']+ str(di_temp['Time'])

            hash_ =hashlib.sha1(st_hash.encode('utf-8')).hexdigest()

            sent_list = sent_tokenize(text)

            for elm in sent_list:
              di_gran={}
              di_gran['hash'] =hash_
              di_gran['combined_text_granular'] =title+'. '+elm
              di_gran['Cat3'] = cat3

              ls_new.append(di_gran)

          temp_cnt+=1

      self.df_new = pd.DataFrame(ls_new)

      total_no = len(self.df_new.axes[0])

      count=0
      for el in unique_labels:
        print(el)

        label_count = len(self.df_new[self.df_new['Cat3']==el].axes[0])
        print(label_count)

        weight_dict[count] = total_no/label_count

        cat_label[el] =count
        count+=1

        print('|'*70)


      X = list(self.df_new['combined_text_granular'])
      Y = [cat_label[el] for el in list(self.df_new['Cat3'])]



    self.X =X
    self.Y =Y
    return  weight_dict, cat_label, len(unique_labels)




  def input_ids_converter(self):

    self.input_ids = []

    for sent in self.X:
      encoded_sent = tokenizer_seq.encode(
                        sent,                      # Sentence to encode.
                        add_special_tokens = True)

      self.input_ids.append(encoded_sent)


    # print('Max sentence length: ', max([len(sen) for sen in self.input_ids]))


    self.input_ids = pad_sequences(self.input_ids, maxlen=MAX_LEN, dtype="long",
                          value=0, truncating="post", padding="post")


  def masking(self):

    self.attention_masks = []

    # For each sentence...
    for sent in self.input_ids:

        att_mask = [int(token_id > 0) for token_id in sent]

        # Store the attention mask for this sentence.
        self.attention_masks.append(att_mask)









In [ ]:
train_df = pd.read_csv(train_csv_path)

prep = preprocessing(train_df)

weight_dict, cat_label, NUM_LABELS = prep.preprocess(0, 'NULL', 'NULL') #-->> 0th level (primary level )preprocessing. Format:- {level, 'NULL', 'NULL'}
# weight_dict, cat_label, NUM_LABELS = prep.preprocess(1, 'NULL', 'beauty') # -->> 1st level (secondary level) preprocessing. Format :-{level, 'NULL', primary_class_name on which secondary classifier is to be made}
# weight_dict, cat_label, NUM_LABELS = prep.preprocess(2, 'NULL', 'NULL') # -->> 2nd level (tertiary level) preprocessing. Format:- {level, 'NULL', 'NULL'}

print(weight_dict)
print(cat_label)

prep.input_ids_converter()
prep.masking()

input_ids =prep.input_ids
att_masks =prep.attention_masks
Y= prep.Y


print(prep.X[0])
print((input_ids[0]))
print(att_masks[0])

In [ ]:
#in case of grocery gourmet food and toys and baby products
prep.df_new.head()

In [ ]:
print(NUM_LABELS)
print(len(input_ids))

splitting data and creating train-val batches in dataloader

In [ ]:
from sklearn.model_selection import train_test_split
# Use 90% for training and 10% for validation in case of level2 classifiers (test_size=0.1).
# Use 80% for training and 20% for validation in case of level1 classifiers (test_size=0.2)
# Use 70% for training and 30% for validation in case of level1 classifiers (test_size =0.3)
train_inputs, validation_inputs, train_labels, validation_labels = train_test_split(input_ids, Y, test_size=0.3, random_state= 2018,stratify=Y)
# Do the same for the masks.
train_masks, validation_masks, _, _ = train_test_split(att_masks, Y, test_size=0.3, random_state=2018,stratify =Y)


# Convert all inputs and labels into torch tensors, the required datatype
# for our model.
train_inputs = torch.tensor(train_inputs)
validation_inputs = torch.tensor(validation_inputs)

train_labels = torch.tensor(train_labels)
validation_labels = torch.tensor(validation_labels)

train_masks = torch.tensor(train_masks)
validation_masks = torch.tensor(validation_masks)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

batch_size = 10

# Create the DataLoader for our training set.
train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

# Create the DataLoader for our validation set.
validation_data = TensorDataset(validation_inputs, validation_masks, validation_labels)
validation_sampler = SequentialSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

Loss, optimizer, definition ;pretrained model loading ; defining learning rates etc



In [ ]:

weights = list(weight_dict.values())

class_weights = torch.FloatTensor(weights).cuda()
print(class_weights)

criterion = torch.nn.CrossEntropyLoss(weight =class_weights)


In [ ]:
model_seq = BertForSequenceClassification.from_pretrained(INPUT_DIR, num_labels= NUM_LABELS)

model_seq.cuda()

In [ ]:
from transformers import AdamW,get_linear_schedule_with_warmup
import numpy as np

import time
import datetime
import random

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

In [ ]:
optimizer = AdamW(model_seq.parameters(),
                  lr = 2e-5, # args.learning_rate - default is 5e-5, our notebook had 2e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
)


# Total number of training steps is number of batches * number of epochs.
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [ ]:
# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def weighted_f1(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return f1_score(labels_flat, preds_flat, average='weighted')


def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))

    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
torch.cuda.empty_cache()

In [ ]:
def forward_pass_and_backprop():
  # Measure how long the training epoch takes.
    t0 = time.time()

    # Reset the total loss for this epoch.
    total_loss = 0

    # Set our model to training mode (as opposed to evaluation mode)
    model_seq.train()

    # This training code is based on the `run_glue.py` script here:
    # https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2d008813037968a9e58/examples/run_glue.py#L128

    # For each batch of training data...
    for step, batch in enumerate(train_dataloader):

        # Progress update every 40 batches.
        if step % 25 == 0 and not step == 0:
            # Calculate elapsed time in minutes.
            elapsed = format_time(time.time() - t0)

            # Report progress.
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        # Put the model into training mode.
        model_seq.train()

        # Unpack this training batch from our dataloader.
        #
        # As we unpack the batch, we'll also copy each tensor to the GPU using the
        # `to` method.
        #
        # `batch` contains three pytorch tensors:
        #   [0]: input ids
        #   [1]: attention masks
        #   [2]: labels
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # Forward pass (evaluate the model on this training batch)
        # `model` is of type: pytorch_pretrained_bert.modeling.BertForSequenceClassification
        outputs = model_seq(b_input_ids,
                    token_type_ids=None,
                    attention_mask=b_input_mask)

        logits =outputs[0]
        loss = criterion(logits, b_labels)

        total_loss += loss.item()

        # Perform a backward pass to calculate the gradients.
        loss.backward()

        # Clip the norm of the gradients to 1.0.
        torch.nn.utils.clip_grad_norm_(model_seq.parameters(), 1.0)

        # Update parameters and take a step using the computed gradient
        optimizer.step()

        # Update the learning rate.
        scheduler.step()

        # Clear out the gradients (by default they accumulate)
        model_seq.zero_grad()

    return total_loss, t0


In [ ]:
# Set the seed value all over the place to make this reproducible.
seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

# Store the average loss after each epoch so we can plot them.
loss_values = []

model_seq.zero_grad()

weighted_f1_max, accuracy_max=0,0
# For each epoch...
for epoch_i in range(0, epochs):
  print("")
  print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
  print('Training...')
  #######################TRAINING ON SINGLE EPOCH###############################
  total_loss, t0= forward_pass_and_backprop()

  # Calculate the average loss over the training data.

  avg_train_loss = total_loss / len(train_dataloader)

  loss_values.append(avg_train_loss)

  print("")
  print("  Average training loss: {0:.2f}".format(avg_train_loss))
  print("  Training epcoh took: {:}".format(format_time(time.time() - t0)))


  ##########################VALIDATION OF TRAINING ON SAME EPOCH#####################

  print("")
  print("Running Validation...")

  t0 = time.time()

  # Put model in evaluation mode to evaluate loss on the validation set
  model_seq.eval()

  # Tracking variables
  eval_loss, eval_accuracy, eval_f1 = 0, 0, 0
  nb_eval_steps, nb_eval_examples = 0, 0

  # Evaluate data for one epoch
  for batch in validation_dataloader:
    # Add batch to GPU
    batch = tuple(t.to(device) for t in batch)

    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels = batch

    # Telling the model not to compute or store gradients, saving memory and speeding up validation
    with torch.no_grad():
      # Forward pass, calculate logit predictions
      # token_type_ids is for the segment ids, but we only have a single sentence here.
      # See https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2d008813037968a9e58/examples/run_glue.py#L258
      outputs = model_seq(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

    logits = outputs[0]

    # Move logits and labels to CPU
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    # Calculate the accuracy for this batch of test sentences.
    tmp_eval_accuracy = flat_accuracy(logits, label_ids)

    # Accumulate the total accuracy.
    eval_accuracy += tmp_eval_accuracy

    #Calculate the f1 for this batch and accumulate it
    temp_weighted_f1 = weighted_f1(logits, label_ids)
    eval_f1 +=temp_weighted_f1

    # Track the number of batches
    nb_eval_steps += 1


  # pred_flat = np.argmax(logits, axis=1).flatten()
  # label_flat =label_ids.flatten()

  # new_f1 =f1_score(label_flat, pred_flat, average='weighted')

  new_f1 = eval_f1/nb_eval_steps
  accuracy = eval_accuracy/nb_eval_steps
  # Report the final accuracy for this validation run.
  print("  Accuracy: {0:.2f}".format(eval_accuracy/nb_eval_steps))
  print("  Validation took: {:}".format(format_time(time.time() - t0)))
  print("f1:-", new_f1)

  if(new_f1 >= weighted_f1_max and accuracy >accuracy_max):
    if(not os.path.exists(OUTPUT_DIR)):
      os.mkdir(OUTPUT_DIR)

    model_seq.save_pretrained(OUTPUT_DIR)
    tokenizer_seq.save_pretrained(OUTPUT_DIR)

    weighted_f1_max =new_f1
    accuracy_max =accuracy

print("")
print("Training complete!")





In [ ]:
import json

filename = open(OUTPUT_DIR+'/'+'cat_label_map.json', 'w')
json.dump(cat_label, filename)
filename.close()

print(cat_label)

In [ ]:
model_test = BertForSequenceClassification.from_pretrained(OUTPUT_DIR)
tokenizer_test =BertTokenizer.from_pretrained(OUTPUT_DIR)


In [ ]:
model_test

In [ ]:
filepath = open(OUTPUT_DIR+'/'+'cat_label_map.json', 'r')
cat_label_test = json.load(filepath)
print(cat_label_test)

In [ ]:
import random

# Set the seed value all over the place to make this reproducible.
seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

# Store the average loss after each epoch so we can plot them.
loss_values = []

model_seq.zero_grad()

# For each epoch...
for epoch_i in range(0, epochs):

    # ========================================
    #               Training
    # ========================================

    # Perform one full pass over the training set.

    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')

    # Measure how long the training epoch takes.
    t0 = time.time()

    # Reset the total loss for this epoch.
    total_loss = 0

    # Set our model to training mode (as opposed to evaluation mode)
    model_seq.train()

    # This training code is based on the `run_glue.py` script here:
    # https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2d008813037968a9e58/examples/run_glue.py#L128

    # For each batch of training data...
    for step, batch in enumerate(train_dataloader):

        # Progress update every 40 batches.
        if step % 40 == 0 and not step == 0:
            # Calculate elapsed time in minutes.
            elapsed = format_time(time.time() - t0)

            # Report progress.
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        # Put the model into training mode.
        model_seq.train()

        # Unpack this training batch from our dataloader.
        #
        # As we unpack the batch, we'll also copy each tensor to the GPU using the
        # `to` method.
        #
        # `batch` contains three pytorch tensors:
        #   [0]: input ids
        #   [1]: attention masks
        #   [2]: labels
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # Forward pass (evaluate the model on this training batch)
        # `model` is of type: pytorch_pretrained_bert.modeling.BertForSequenceClassification
        outputs = model_seq(b_input_ids,
                    token_type_ids=None,
                    attention_mask=b_input_mask,
                    labels=b_labels)

        loss = outputs[0]

        # Accumulate the loss.

        # Accumulate the loss. `loss` is a Tensor containing a single value;
        # the `.item()` function just returns the Python value from the tensor.
        total_loss += loss.item()

        # Perform a backward pass to calculate the gradients.
        loss.backward()

        # Clip the norm of the gradients to 1.0.
        torch.nn.utils.clip_grad_norm_(model_seq.parameters(), 1.0)

        # Update parameters and take a step using the computed gradient
        optimizer.step()

        # Update the learning rate.
        scheduler.step()

        # Clear out the gradients (by default they accumulate)
        model_seq.zero_grad()

    # Calculate the average loss over the training data.
    avg_train_loss = total_loss / len(train_dataloader)

    loss_values.append(avg_train_loss)

    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epcoh took: {:}".format(format_time(time.time() - t0)))

    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set.

    print("")
    print("Running Validation...")

    t0 = time.time()

    # Put model in evaluation mode to evaluate loss on the validation set
    model_seq.eval()

    # Tracking variables
    eval_loss, eval_accuracy = 0, 0
    nb_eval_steps, nb_eval_examples = 0, 0

    # Evaluate data for one epoch
    for batch in validation_dataloader:

        # Add batch to GPU
        batch = tuple(t.to(device) for t in batch)

        # Unpack the inputs from our dataloader
        b_input_ids, b_input_mask, b_labels = batch

        # Telling the model not to compute or store gradients, saving memory and speeding up validation
        with torch.no_grad():
            # Forward pass, calculate logit predictions
            # token_type_ids is for the segment ids, but we only have a single sentence here.
            # See https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2d008813037968a9e58/examples/run_glue.py#L258
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

        logits = outputs[0]

        # Move logits and labels to CPU
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        # Calculate the accuracy for this batch of test sentences.
        tmp_eval_accuracy = flat_accuracy(logits, label_ids)

        # Accumulate the total accuracy.
        eval_accuracy += tmp_eval_accuracy

        # Track the number of batches
        nb_eval_steps += 1

    # Report the final accuracy for this validation run.
    print("  Accuracy: {0:.2f}".format(eval_accuracy/nb_eval_steps))
    print("  Validation took: {:}".format(format_time(time.time() - t0)))

print("")
print("Training complete!")

In [ ]:
print(os.path.exists(OUTPUT_DIR))